# Kernels as functions

<div class="alert alert-info">

**Summary**

This lesson reads the kernels of the consumption stage of the portfolio–consumption model as plain Python functions. It prints the source text of the value aggregator's function, evaluates that function and the function of the stated derivative on a grid, composes one step of the decision builder by hand from the law of motion, the aggregator and the derivative, compiles the same functions with Numba, passes the values of a second calibration to the compiled code without compiling it again, and reads the record of a builder's kernels that a solver takes. The results of the step are kept in one dictionary per perch. The lesson evaluates kernels and solves nothing.

</div>

<div class="alert alert-success">

**Relationship with other documents**

**Builds on** — [Value functions and the backward builders](../getting-started/02-value-functions.ipynb) and [Parameter values, methods and settings](../getting-started/04-parameters-methods-and-settings.ipynb): the decision builder, its kernels and a calibrated version.

**Formal account** — [A piece](../../docs/Bellman-Sym/API/stage/05-piece.md) §2 and [A builder](../../docs/Bellman-Sym/API/stage/04-builder.md) §5: what `kernel()`, `compile_kernel()` and `kernels()` answer.

**See also** — [Solutions at the perches](../../docs/Bellman-Sym/11-solutions-at-the-perches.md): the dictionaries in which a solver keeps its arrays; the model's numerical tool, `applications/cons-port/code/cons_port_tool.py`, which builds its endogenous-grid step from two records of kernels.

</div>

<div class="alert">

**Key terminology**

| Term | Meaning |
|---|---|
| kernel | The mathematical data an operation uses: a law of motion for a pullback, a value aggregator for an aggregation, the right side of a derivative line for the piece that states a derivative. |
| source text | The text of the Python function that computes a kernel, which `kernel()` holds under `python`. |
| parameter values | The numbers a calibration binds to the parameters a kernel reads, passed to its function at each call. |
| Numba | A compiler for numerical Python functions: `numba.njit(f)` answers a version of `f` that is compiled to machine code at its first call, once for each combination of argument types. |
| [[perch]] | One of the three information sets of a stage: arrival `arvl`, decision `dcsn` and continuation `cntn`. |
| record of kernels | What a builder's `kernels()` answers: its kernels in the order they are applied, with their functions and parameter values, and no reference to the stage. |

</div>

The [[stage]] is the consumption stage of the getting-started lessons: resources $\mathrm m_d$ at the decision, consumption $\mathrm c \in (0, \mathrm m_d]$, savings $\mathrm a = \mathrm m_d - \mathrm c$ at the continuation, utility $u(\mathrm c) = \mathrm c^{1-\rho}/(1-\rho)$ and marginal utility $\mathit{du}(\mathrm c) = \mathrm c^{-\rho}$. Its backward decision builder, `cntn_to_dcsn`, reads the continuation value $V_\succ$, a function of savings, and at a consumption rule $c$ gives the value and the marginal value at the decision,

$$
V(\mathrm m_d) = u\big(c(\mathrm m_d)\big) + V_\succ\big(g(\mathrm m_d, c(\mathrm m_d))\big), \qquad dV(\mathrm m_d) = \mathit{du}\big(c(\mathrm m_d)\big),
$$

by three kernels: the law of motion $g(\mathrm m_d, \mathrm c) = \mathrm m_d - \mathrm c$, which its pullback applies; the aggregator $G(V_\succ, \mathrm m_d, \mathrm c) = u(\mathrm c) + V_\succ$, which its aggregation applies; and the right side of the file's derivative line, $\mathit{du}(\mathrm c)$, which the piece of the stated derivative applies.

The lesson writes only the two files of the interactive view of the stage opened in the lesson's first section, in the folder model-explorer beside it, and reads two files:

- `applications/portfolio-choice/declaration/stages/cons/cons.bl`
- `applications/cons-port/declaration/calibration/portfolio.yml`

The first code cell imports the collection's set-up module, checkout, which the first lesson describes, then Numba, NumPy and the two functions of the package the lesson calls; it constructs the consumption stage from its file and calibrates it by the model's calibration file, which sets $\rho = 6$.

In [1]:
import checkout
import numba
import numpy as np
from bellman.stage import elaborate_stage, load_stage
cons = elaborate_stage(load_stage("applications/portfolio-choice/declaration/stages/cons/cons.bl")).calibrate(
    "applications/cons-port/declaration/calibration/portfolio.yml")

## The kernel and its text

The first code cell named the calibrated version `cons`. `kernel()` answers the kernel an operation applies as a record, displayed as its printed rule. The next cell asks the aggregation of the decision builder for its kernel and names the record `aggregator`.

In [2]:
aggregator = cons.dcsn.builder.bwd["Aggregation"].kernel()
aggregator

𝐺 of cntn_to_dcsn: 𝐺(V≻, m_d, c) = 𝑢(c) + V≻
output V; unused m_d

`checkout.explore` shows the stage in [Google's Model Explorer](https://github.com/google-ai-edge/model-explorer), an interactive viewer in which each builder is a frame opened to the operations it holds, and in which selecting the aggregation of `cntn_to_dcsn`, the operation labeled `Aggregation at 𝑐` over `𝐺`, shows in a side panel the two lines of its kernel displayed above, as the rows `kernel 1` and `kernel 2`. The view opens only when the lesson is run: the saved output of the next cell is the paths of the two files it writes into the folder `model-explorer` beside the lesson.

In [3]:
checkout.explore(cons)

tutorials/topics/model-explorer/cons-calibrated.graph.json
tutorials/topics/model-explorer/cons-calibrated.edge-overlays.json


$G$ reads the continuation value, resources and consumption and returns the value $V$; it does not use $\mathrm m_d$. The marginal value $dV$ is stated by the file's derivative line, `D_{m_d}V = 𝑑𝑢(c)`, and is a piece of its own in the builder's record of kernels, read below.

Besides the rule, the record holds `parameters`, the names of the parameters the kernel reads, here `('ρ',)`; `parameter_values`, the values this version's calibration binds to them, `(6.0,)`; and `python`, the source text of one plain Python function, which the next cell prints.

In [4]:
print(aggregator.python)

# bellman kernel of cons, cntn_to_dcsn:Aggregation; cons.bl sha256 e60110b0b964ce482812c558922bc7aa6eaaf14421e2e162b9f87cc7b120a667; bellman 0.1.7
# 𝐺 of cntn_to_dcsn: 𝐺(V≻, m_d, c) = 𝑢(c) + V≻
# output V; unused m_d
# parameters: ρ ∈ [1, inf)
# names: V_cntn is V≻; u is 𝑢
import numpy as np

def G_cntn_to_dcsn(V_cntn, m_d, c, ρ):
    def u(c):
        return c**(1 - ρ)/(1 - ρ)
    V = u(c) + V_cntn
    return V



The comment lines name the stage, the operation, the stage file with the digest of its text, the package's version, the printed rule, the Python name given to each name of the file and the declared set of $\rho$. They hold no date and no value, so the text of an unchanged model does not change. The function's arguments are the kernel's own, in the printed order, followed by $\rho$; $u$ is written inside it, and NumPy, `np`, is the only name it reads from outside.

## The function on a grid

`compile_kernel()` executes that text and answers the function it defines, which the next cell names `G`. The cell evaluates $G$ on ten levels of savings from 0.5 to 5, named `a`, with no continuation value, $V_\succ = 0$, and with consumption equal to resources, $\mathrm c = \mathrm m_d = \mathrm a$: this is the value $u(\mathrm a)$ of consuming all of one's savings, named `v`. The calibration's values are passed at the call, after the kernel's own arguments.

In [5]:
G = cons.dcsn.builder.bwd["Aggregation"].compile_kernel()
a = np.linspace(0.5, 5.0, 10)
v = G(np.zeros_like(a), a, a, *aggregator.parameter_values)
v

array([-6.40000000e+00, -2.00000000e-01, -2.63374486e-02, -6.25000000e-03,
       -2.04800000e-03, -8.23045267e-04, -3.80793717e-04, -1.95312500e-04,
       -1.08384562e-04, -6.40000000e-05])

The marginal value is computed by the piece of the stated derivative, `marginal:Derivative:1`, one of the pieces of the builder's record of kernels, `kernels()`, which the last section reads in full: its function takes the aggregation's arguments and returns $\mathit{du}(\mathrm c)$, and it holds its own `parameter_values`, the same `(6.0,)`. The next cell takes that piece from the record by its key, names its function `D`, and evaluates it at the same points as `dv`, the marginal value $\mathit{du}(\mathrm a)$ of consuming all of one's savings.

In [6]:
D = next(piece for piece in cons.dcsn.builder.bwd.kernels().pieces if piece.key == "marginal:Derivative:1").function
dv = D(np.zeros_like(a), a, a, *aggregator.parameter_values)
dv

array([6.40000000e+01, 1.00000000e+00, 8.77914952e-02, 1.56250000e-02,
       4.09600000e-03, 1.37174211e-03, 5.43991024e-04, 2.44140625e-04,
       1.20427291e-04, 6.40000000e-05])

At $\rho = 6$, $u(1) = -0.2$ and $\mathit{du}(1) = 1$, the second entries of `v` and `dv`.

## One step of the decision builder

The next cells take `v` as the continuation value $V_\succ$ on the savings grid `a`: the value of the savings carried into the next stage when all of them are consumed there. `compile_kernel()` on the pullback answers the law of motion $g$, which the next cell names `g`. On resources `m_d`, twice the savings grid, and at the rule $c(\mathrm m_d) = \mathrm m_d/2$, the cell composes one step of the decision builder by hand: $g$ gives the savings each level of resources leaves, `np.interp` reads the continuation value there from the grid, $G$ adds utility and returns the value at the decision, and $D$ gives the marginal value there.

In [7]:
g = cons.dcsn.builder.bwd["Pullback"].compile_kernel()
m_d = 2 * a
(G(np.interp(g(m_d, m_d / 2), a, v), m_d, m_d / 2, *aggregator.parameter_values),
 D(np.interp(g(m_d, m_d / 2), a, v), m_d, m_d / 2, *aggregator.parameter_values))

(array([-1.28000000e+01, -4.00000000e-01, -5.26748971e-02, -1.25000000e-02,
        -4.09600000e-03, -1.64609053e-03, -7.61587434e-04, -3.90625000e-04,
        -2.16769124e-04, -1.28000000e-04]),
 array([6.40000000e+01, 1.00000000e+00, 8.77914952e-02, 1.56250000e-02,
        4.09600000e-03, 1.37174211e-03, 5.43991024e-04, 2.44140625e-04,
        1.20427291e-04, 6.40000000e-05]))

Each value is twice the continuation value at the same position and each marginal value equals `dv`: consuming half of the resources leaves the other half as savings, which lie on the grid, so `np.interp` reads the continuation value there exactly. The rule maximizes $u(\mathrm c) + u(\mathrm m_d - \mathrm c)$, which is concave and symmetric about $\mathrm c = \mathrm m_d/2$; the step evaluates the rule and chooses nothing. The step is written as two expressions over the three functions; the next cell compiles each of them.

## The same step compiled

`numba.njit` compiles each kernel in one call: $u$ and $\mathit{du}$ are inside the kernels' text, so no other function needs compiling first. The next cell compiles the three kernels as `g_jit`, `G_jit` and `D_jit`, repeats the step with them, and keeps the results by [[perch]] and field in the form [Solutions at the perches](../../docs/Bellman-Sym/11-solutions-at-the-perches.md) describes: a dictionary for each perch, `cntn` and `dcsn` as the stage file marks them, each holding its grid under the name of its coordinate, its arrays under the names the file gives the fields, and the derivative of the value under `dV`, as that page names it.

In [8]:
g_jit, G_jit, D_jit = numba.njit(g), numba.njit(G), numba.njit(D)
V = G_jit(np.interp(g_jit(m_d, m_d / 2), a, v), m_d, m_d / 2, *aggregator.parameter_values)
dV = D_jit(np.interp(g_jit(m_d, m_d / 2), a, v), m_d, m_d / 2, *aggregator.parameter_values)
solution = {"cntn": {"a": a, "V": v, "dV": dv},
            "dcsn": {"m_d": m_d, "c": m_d / 2, "V": V, "dV": dV}}
solution

{'cntn': {'a': array([0.5, 1. , 1.5, 2. , 2.5, 3. , 3.5, 4. , 4.5, 5. ]),
  'V': array([-6.40000000e+00, -2.00000000e-01, -2.63374486e-02, -6.25000000e-03,
         -2.04800000e-03, -8.23045267e-04, -3.80793717e-04, -1.95312500e-04,
         -1.08384562e-04, -6.40000000e-05]),
  'dV': array([6.40000000e+01, 1.00000000e+00, 8.77914952e-02, 1.56250000e-02,
         4.09600000e-03, 1.37174211e-03, 5.43991024e-04, 2.44140625e-04,
         1.20427291e-04, 6.40000000e-05])},
 'dcsn': {'m_d': array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.]),
  'c': array([0.5, 1. , 1.5, 2. , 2.5, 3. , 3.5, 4. , 4.5, 5. ]),
  'V': array([-1.28000000e+01, -4.00000000e-01, -5.26748971e-02, -1.25000000e-02,
         -4.09600000e-03, -1.64609053e-03, -7.61587434e-04, -3.90625000e-04,
         -2.16769124e-04, -1.28000000e-04]),
  'dV': array([6.40000000e+01, 1.00000000e+00, 8.77914952e-02, 1.56250000e-02,
         4.09600000e-03, 1.37174211e-03, 5.43991024e-04, 2.44140625e-04,
         1.20427291e-04, 6.4

The compiled step gives the numbers of the step by hand. The stage holds none of them: `solution` is a plain dictionary of NumPy arrays.

## A second calibration

`calibrate` binds the stage it is called on and returns it. The next cell calibrates `cons` again at $\rho = 2$, so that the record the last section prints holds that value, and names the aggregation's kernel there `aggregator_ρ2`. It then shows, in order, whether the two source texts are equal, the new parameter values, `G_jit` evaluated with them on the savings grid, and how many compiled versions of `G_jit` Numba holds.

In [9]:
aggregator_ρ2 = cons.calibrate({"ρ": 2.0}).dcsn.builder.bwd["Aggregation"].kernel()
(aggregator_ρ2.python == aggregator.python, aggregator_ρ2.parameter_values,
 G_jit(np.zeros_like(a), a, a, *aggregator_ρ2.parameter_values), len(G_jit.signatures))

(True,
 (2.0,),
 array([-2.        , -1.        , -0.66666667, -0.5       , -0.4       ,
        -0.33333333, -0.28571429, -0.25      , -0.22222222, -0.2       ]),
 1)

The text is unchanged and the value of $\rho$ is an argument of the call, so the function Numba compiled at its first call, a call with $\rho = 6$, gives the values at $\rho = 2$, $u(1) = -1$ among them, with no second compilation: Numba holds one compiled version.

## The record a solver reads

A builder answers `kernels()`: one record of the kernels of its operations in the order they are applied, with the piece of each stated derivative after them, which refers to no stage. The next cell prints the record of the decision builder.

In [10]:
cons.dcsn.builder.bwd.kernels()

kernels of cntn_to_dcsn: 𝑉≻ → 𝑉
1  Pullback
     function  g_dcsn_to_cntn(m_d, c) → a≻
     wires     𝑉≻ → 𝑉≻∘𝑔(·, 𝑐)
2  Aggregation
     function  G_cntn_to_dcsn(V≻, m_d, c; ρ) → V
     wires     𝑉≻∘𝑔(·, 𝑐) → 𝑉
3  Evaluation
     kernel    identity at m_d; no policy is inserted
     wires     𝑉 → 𝑉
4  marginal:Derivative:1
     function  D_cntn_to_dcsn(V≻, m_d, c; ρ) → D_{m_d}V
     wires     V≻, m_d, c → D_{m_d}V
parameters   ρ = 2.0
sets         m_d ∈ Xm, c ∈ ℝ₊₊, V≻ ∈ [-inf, inf), Xm := ℝ₊₊

The record holds, in the order of application, the pullback with $g$, the aggregation with $G$, the evaluation, which applies no function and is held as its printed rule, and the piece of the stated derivative with $D$, each with the names of its arguments and outputs and the wires it reads and writes; then the parameters with their values and the declared sets. A solver takes from it what the step above took by hand: the three functions, the order of their arguments and the values to pass. The model's numerical tool builds its endogenous-grid step by one call of `egm` from [asva](../../docs/asva/index.md), the numerical toolkit beside the package `bellman`, on the record of the policy builder and the record of this builder without its derivative piece, which the tool evaluates itself, with the savings and evaluation grids the settings define; the operator the call returns reads no stage.

## What was checked and what is assumed

Checked by the displays: the kernel's source text; the values of its function and of the derivative piece's function on a grid; one step of the decision builder composed by hand, and the same numbers from the functions Numba compiled; a second calibration passed to the compiled function without a second compilation. Assumed: that $\mathit{du}$ is the derivative of $u$, which the stage file declares and the package does not verify, and the continuation value `v`, which the lesson chose.